**Restaurant Review Classification**

Dataset: https://www.kaggle.com/datasets/d4rklucif3r/restaurant-reviews

In [ ]:
import pickle
import random

import nltk
import pandas as pd

# Bigrams
from nltk import bigrams

# Stopwords
from nltk.corpus import stopwords

# Frequency Distribution
from nltk.probability import FreqDist

# Tokenize
from nltk.tokenize import word_tokenize

# Classifier
from nltk.classify import SklearnClassifier, accuracy
from sklearn.linear_model import LogisticRegression

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

In [ ]:
# Shared preprocessing helpers
NEGATION = {"not", "no", "nor", "don", "t", "didn", "wasn", "isn"}
STOPS = set(stopwords.words("english")) - NEGATION


def keep(w):
    # stripping the apostrophe lets "n't" and "did n't" pass the alpha check
    return w not in STOPS and w.replace(" ", "").replace("'", "").isalpha()


def toFeat(toks, vocab):
    s = set(toks)
    return {v: (v in s) for v in vocab}


def tokensWithBigrams(text):
    tokens = word_tokenize(text.lower())
    return tokens + [" ".join(b) for b in bigrams(tokens)]

In [ ]:
# MENU
def mainMenu():
    while True:
        print("1. Build Model")
        print("2. Show Model Evaluation")
        print("3. Test Review")
        print("4. Exit")

        try:
            choice = int(input(">> "))
            if choice >= 1 and choice <= 4:
                return choice
            else:
                print("Input must be between [1-4]")
        except ValueError:
            print("Input must be an integer")

In [ ]:
### CREATE/BUILD MODEL ###
def buildModel(df, vocabSize=200):
    rows = [
        (tokensWithBigrams(r), "Positive" if l == 1 else "Negative")
        for r, l in zip(df["Review"], df["Liked"])
    ]

    random.Random(42).shuffle(rows)
    cut = int(len(rows) * 0.7)
    train, test = rows[:cut], rows[cut:]

    # Vocabulary from TRAIN only (no leakage)
    words = [w for toks, _ in train for w in toks if keep(w)]
    vocab = [w for w, _ in FreqDist(words).most_common(vocabSize)]

    training_data = [(toFeat(t, vocab), l) for t, l in train]
    testing_data = [(toFeat(t, vocab), l) for t, l in test]

    classifier = SklearnClassifier(LogisticRegression(max_iter=1000)).train(
        training_data
    )

    tp = 0  # True positives
    fp = 0  # False positives
    fn = 0  # False negatives

    for features, label in testing_data:
        predicted_label = classifier.classify(features)
        if predicted_label == "Positive" and label == "Positive":
            tp += 1
        elif predicted_label == "Positive" and label == "Negative":
            fp += 1
        elif predicted_label == "Negative" and label == "Positive":
            fn += 1

    pos_precision = tp / (tp + fp) if (tp + fp) else 0
    pos_recall = tp / (tp + fn) if (tp + fn) else 0

    print(f"TP={tp}, FP={fp}, FN={fn}")
    print(f"Precision (Positive): {pos_precision:.4f}")
    print(f"Recall (Positive): {pos_recall:.4f}")
    print(f"Accuracy: {accuracy(classifier, testing_data):.4f}\n")

    # Save Model
    with open("../result/model.pickle", "wb") as file:
        pickle.dump(classifier, file)

    with open("../result/test.pickle", "wb") as file:
        pickle.dump(testing_data, file)

    with open("../result/vocab.pickle", "wb") as file:
        pickle.dump(vocab, file)

In [ ]:
### READ MODEL ###
def readModel():
    with open("../result/model.pickle", "rb") as file:
        classifier = pickle.load(file)
        return classifier


### Input Review ###
def reviewTest():
    classifier = readModel()
    with open("../result/vocab.pickle", "rb") as file:
        vocab = pickle.load(file)
    review = input("Input review here: ")
    features = toFeat(tokensWithBigrams(review), vocab)
    print(f"{classifier.classify(features)}\n")


### Show Accuracy ###
def showAccuracy():
    tp = 0  # True positives
    fp = 0  # False positives
    fn = 0  # False negatives

    with open("../result/test.pickle", "rb") as file:
        testing_data = pickle.load(file)
        classifier = readModel()

        for features, label in testing_data:
            predicted_label = classifier.classify(features)
            if predicted_label == "Positive" and label == "Positive":
                tp += 1
            elif predicted_label == "Positive" and label == "Negative":
                fp += 1
            elif predicted_label == "Negative" and label == "Positive":
                fn += 1

        pos_precision = tp / (tp + fp) if (tp + fp) else 0
        pos_recall = tp / (tp + fn) if (tp + fn) else 0

        print(f"TP={tp}, FP={fp}, FN={fn}")
        print(f"Precision (Positive): {pos_precision:.4f}")
        print(f"Recall (Positive): {pos_recall:.4f}")
        print(f"Accuracy: {accuracy(classifier, testing_data):.4f}\n")

In [ ]:
def tuneVocab(df, sizes=(200, 500, 1000, 1200, 1500), seeds=(5, 25, 42)):
    # Tokenize each review once
    reviews = [
        (tokensWithBigrams(r), "Positive" if l == 1 else "Negative")
        for r, l in zip(df["Review"], df["Liked"])
    ]

    results = {}
    for size in sizes:
        accs = []
        for seed in seeds:
            rows = reviews[:]
            random.Random(seed).shuffle(rows)
            cut = int(len(rows) * 0.7)
            train, test = rows[:cut], rows[cut:]

            # Build vocabulary from TRAIN only (avoids leakage)
            words = [w for toks, _ in train for w in toks if keep(w)]

            vocab = [w for w, _ in FreqDist(words).most_common(size)]

            trainSet = [(toFeat(t, vocab), l) for t, l in train]
            testSet = [(toFeat(t, vocab), l) for t, l in test]

            clf = SklearnClassifier(LogisticRegression(max_iter=1000)).train(trainSet)
            accs.append(accuracy(clf, testSet))

        results[size] = sum(accs) / len(accs)
        print(f"vocab={size:5d}  avg accuracy={results[size]:.4f}")

    best = max(results, key=results.get)
    print(f"\nBest vocab size: {best} ({results[best]:.4f})")
    return best

In [ ]:
df = pd.read_csv("../data/Restaurant_Reviews.tsv", sep="\t").dropna()
best = tuneVocab(df)

In [ ]:
### MAIN MENU ###
menu = 0

while True:
    menu = mainMenu()
    if menu == 1:
        print("Building Model and Evaluation...")
        buildModel(df, best)
    elif menu == 2:
        print("Model Evaluation")
        showAccuracy()
    elif menu == 3:
        print("Test Review")
        reviewTest()
    elif menu == 4:
        print("Goodbye...")
        break